# Addresses26 — Agent Attribution v4 Manual Review Workflow

Purpose:
- Use the verified Land Registry BN sales dataset as the factual core.
- Create a controlled manual review queue for estate-agent attribution.
- Generate helpful search URLs for each transaction.
- Allow you to manually fill `manual_agent_name`, `source_url`, `confidence`, and notes.
- Re-ingest manual edits and produce candidate attribution tables and summaries.

This notebook does **not** scrape portals or bypass restrictions. It creates a supervised evidence workflow.

In [1]:
# --- 1. Setup ---

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone
from urllib.parse import quote_plus

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

BASE = Path('/content/drive/MyDrive/Addresses26_Data')
DERIVED = BASE / 'derived' / 'land_registry_price_paid'
REPORTS = BASE / 'reports' / 'agent_attribution'
REPORTS.mkdir(parents=True, exist_ok=True)

print('BASE   :', BASE)
print('DERIVED:', DERIVED)
print('REPORTS:', REPORTS)

Mounted at /content/drive
BASE   : /content/drive/MyDrive/Addresses26_Data
DERIVED: /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid
REPORTS: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution


In [2]:
# --- 2. Configuration ---

POSTCODE_PREFIX = 'BN'
YEARS = list(range(2021, 2026))  # last 5 completed / near-completed years for the pilot
SAMPLE_PER_YEAR = 50
RANDOM_SEED = 26

# Use True if you want a balanced district sample where possible.
BALANCE_BY_DISTRICT = True

print('Years:', YEARS)
print('Sample per year:', SAMPLE_PER_YEAR)

Years: [2021, 2022, 2023, 2024, 2025]
Sample per year: 50


In [3]:
# --- 3. Load verified Land Registry parquet files ---

parquet_files = []
for year in YEARS:
    p = DERIVED / f'pp_bn_standard_{year}.parquet'
    if p.exists():
        parquet_files.append(p)
    else:
        print('Missing parquet:', p)

assert parquet_files, 'No parquet files found. Check DERIVED path and filenames.'

frames = []
for p in parquet_files:
    df = pd.read_parquet(p)
    df['source_file'] = p.name
    frames.append(df)

sales = pd.concat(frames, ignore_index=True)

# Normalise key fields
sales.columns = [str(c).strip().lower() for c in sales.columns]
if 'date_of_transfer' not in sales.columns and 'date' in sales.columns:
    sales = sales.rename(columns={'date': 'date_of_transfer'})

sales['postcode'] = sales['postcode'].astype(str).str.upper().str.strip()
sales['date_of_transfer'] = pd.to_datetime(sales['date_of_transfer'], errors='coerce')
sales['sale_year'] = sales['date_of_transfer'].dt.year
sales['price'] = pd.to_numeric(sales['price'], errors='coerce')

# Basic verified-core filters
sales = sales[
    sales['postcode'].str.startswith(POSTCODE_PREFIX, na=False)
    & sales['sale_year'].isin(YEARS)
    & sales['price'].notna()
    & sales['date_of_transfer'].notna()
].copy()

if 'ppd_category_type' in sales.columns:
    sales = sales[sales['ppd_category_type'].astype(str).str.upper().eq('A')].copy()

# postcode district
sales['postcode_district'] = sales['postcode'].str.extract(r'^([A-Z]{1,2}\d{1,2}[A-Z]?)', expand=False)

print('Loaded sales rows:', len(sales))
display(sales.head())
display(sales.groupby('sale_year').size().reset_index(name='rows'))

Loaded sales rows: 68685


,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category_type,record_status,postcode_area,postcode_district,postcode_sector,property_type_label,old_new_label,duration_label,address_compact,source_file,sale_year
0,{F87E72F8-D9D5-176C-E053-6B04A8C0D2BE},650000,2021-01-03,BN3 5DL,S,N,F,58,<NA>,PORTLAND ROAD,<NA>,HOVE,BRIGHTON AND HOVE,BRIGHTON AND HOVE,A,A,BN,BN3,None,Semi-detached,Established,Freehold,"58, PORTLAND ROAD, HOVE",pp_bn_standard_2021.parquet,2021
1,{BC8936BB-74DB-0E2C-E053-6C04A8C0DBF4},265000,2021-01-04,BN1 3RT,F,N,L,1,SECOND FLOOR FLAT,WEST HILL ROAD,<NA>,BRIGHTON,BRIGHTON AND HOVE,BRIGHTON AND HOVE,A,A,BN,BN1,None,Flat/Maisonette,Established,Leasehold,"SECOND FLOOR FLAT, 1, WEST HILL ROAD, BRIGHTON",pp_bn_standard_2021.parquet,2021
2,{BC8936BB-F58B-0E2C-E053-6C04A8C0DBF4},505000,2021-01-04,BN16 2EA,D,N,F,45,<NA>,GLENVILLE ROAD,RUSTINGTON,LITTLEHAMPTON,ARUN,WEST SUSSEX,A,A,BN,BN16,None,Detached,Established,Freehold,"45, GLENVILLE ROAD, RUSTINGTON, LITTLEHAMPTON",pp_bn_standard_2021.parquet,2021
3,{BC8936BB-743D-0E2C-E053-6C04A8C0DBF4},430000,2021-01-04,BN2 5AD,F,N,L,"THE LEAS, 34 - 35",FLAT 1,SUSSEX SQUARE,<NA>,BRIGHTON,BRIGHTON AND HOVE,BRIGHTON AND HOVE,A,A,BN,BN2,None,Flat/Maisonette,Established,Leasehold,"FLAT 1, THE LEAS, 34 - 35, SUSSEX SQUARE, BRIG...",pp_bn_standard_2021.parquet,2021
4,{BC8936BB-6FA6-0E2C-E053-6C04A8C0DBF4},800000,2021-01-04,BN20 8DL,D,N,F,25,<NA>,OLD CAMP ROAD,<NA>,EASTBOURNE,EASTBOURNE,EAST SUSSEX,A,A,BN,BN20,None,Detached,Established,Freehold,"25, OLD CAMP ROAD, EASTBOURNE",pp_bn_standard_2021.parquet,2021


,sale_year,rows
0,2021,18279
1,2022,14934
2,2023,11515
3,2024,12532
4,2025,11425


In [4]:
# --- 4. Create stable transaction keys and useful address strings ---

# Land Registry columns vary depending on ingestion. These are common PPD fields.
address_cols = [
    'paon', 'saon', 'street', 'locality', 'town_city', 'district', 'county', 'postcode'
]
available_address_cols = [c for c in address_cols if c in sales.columns]

def clean_part(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.lower() in {'nan', 'none', '<na>'}:
        return ''
    return s

def build_address(row):
    parts = [clean_part(row.get(c, '')) for c in available_address_cols]
    parts = [p for p in parts if p]
    return ', '.join(parts)

sales['address_text'] = sales.apply(build_address, axis=1)

# Stable key: enough for review and later joins, without relying on title number.
key_cols = ['price', 'date_of_transfer', 'postcode']
for c in ['paon', 'saon', 'street']:
    if c in sales.columns:
        key_cols.append(c)

sales['transaction_key'] = (
    sales[key_cols]
    .astype(str)
    .agg('|'.join, axis=1)
    .str.upper()
    .str.replace(r'\s+', ' ', regex=True)
)

print('Available address columns:', available_address_cols)
display(sales[['transaction_key', 'address_text', 'price', 'date_of_transfer', 'postcode', 'postcode_district']].head())

Available address columns: ['paon', 'saon', 'street', 'locality', 'town_city', 'district', 'county', 'postcode']


,transaction_key,address_text,price,date_of_transfer,postcode,postcode_district
0,650000|2021-01-03|BN3 5DL|58|<NA>|PORTLAND ROAD,"58, PORTLAND ROAD, HOVE, BRIGHTON AND HOVE, BR...",650000,2021-01-03,BN3 5DL,BN3
1,265000|2021-01-04|BN1 3RT|1|SECOND FLOOR FLAT|...,"1, SECOND FLOOR FLAT, WEST HILL ROAD, BRIGHTON...",265000,2021-01-04,BN1 3RT,BN1
2,505000|2021-01-04|BN16 2EA|45|<NA>|GLENVILLE ROAD,"45, GLENVILLE ROAD, RUSTINGTON, LITTLEHAMPTON,...",505000,2021-01-04,BN16 2EA,BN16
3,"430000|2021-01-04|BN2 5AD|THE LEAS, 34 - 35|FL...","THE LEAS, 34 - 35, FLAT 1, SUSSEX SQUARE, BRIG...",430000,2021-01-04,BN2 5AD,BN2
4,800000|2021-01-04|BN20 8DL|25|<NA>|OLD CAMP ROAD,"25, OLD CAMP ROAD, EASTBOURNE, EASTBOURNE, EAS...",800000,2021-01-04,BN20 8DL,BN20


In [5]:
# --- 5. Build a balanced pilot sample ---

rng = np.random.default_rng(RANDOM_SEED)

if BALANCE_BY_DISTRICT:
    samples = []
    # First sample by year and district to spread across BN.
    for year, df_year in sales.groupby('sale_year'):
        districts = sorted(df_year['postcode_district'].dropna().unique())
        if not districts:
            continue
        per_district = max(1, SAMPLE_PER_YEAR // len(districts))
        chunks = []
        for district, df_dist in df_year.groupby('postcode_district'):
            n = min(per_district, len(df_dist))
            chunks.append(df_dist.sample(n=n, random_state=RANDOM_SEED + int(year) + len(str(district))))
        sampled = pd.concat(chunks, ignore_index=True) if chunks else df_year.head(0)
        if len(sampled) < SAMPLE_PER_YEAR:
            remaining = df_year[~df_year['transaction_key'].isin(sampled['transaction_key'])]
            topup_n = min(SAMPLE_PER_YEAR - len(sampled), len(remaining))
            if topup_n > 0:
                sampled = pd.concat([
                    sampled,
                    remaining.sample(n=topup_n, random_state=RANDOM_SEED + int(year))
                ], ignore_index=True)
        samples.append(sampled)
    review = pd.concat(samples, ignore_index=True)
else:
    review = (
        sales.groupby('sale_year', group_keys=False)
        .apply(lambda g: g.sample(n=min(SAMPLE_PER_YEAR, len(g)), random_state=RANDOM_SEED + int(g.name)))
        .reset_index(drop=True)
    )

review = review.drop_duplicates('transaction_key').reset_index(drop=True)

print('Review rows:', len(review))
display(review.groupby('sale_year').size().reset_index(name='sample_rows'))
display(review.groupby('postcode_district').size().reset_index(name='sample_rows').sort_values('sample_rows', ascending=False))

Review rows: 250


,sale_year,sample_rows
0,2021,50
1,2022,50
2,2023,50
3,2024,50
4,2025,50


,postcode_district,sample_rows
19,BN3,17
6,BN15,12
14,BN23,11
10,BN2,11
5,BN14,10
3,BN12,10
0,BN1,10
4,BN13,9
13,BN22,9
1,BN10,9


In [6]:
# --- 6. Generate manual-review search URLs ---

def make_query(row, portal=None):
    address = row.get('address_text', '')
    postcode = row.get('postcode', '')
    price = int(row['price']) if pd.notna(row.get('price')) else ''
    year = int(row['sale_year']) if pd.notna(row.get('sale_year')) else ''
    base = f'"{postcode}" "{price}" "{year}" property sold estate agent'
    if address:
        base = f'"{address}" "{price}" "{year}" estate agent'
    if portal:
        base += f' site:{portal}'
    return base

def google_url(query):
    return 'https://www.google.com/search?q=' + quote_plus(query)

def bing_url(query):
    return 'https://www.bing.com/search?q=' + quote_plus(query)

review['google_search_url'] = review.apply(lambda r: google_url(make_query(r)), axis=1)
review['rightmove_search_url'] = review.apply(lambda r: google_url(make_query(r, 'rightmove.co.uk')), axis=1)
review['zoopla_search_url'] = review.apply(lambda r: google_url(make_query(r, 'zoopla.co.uk')), axis=1)
review['onthemarket_search_url'] = review.apply(lambda r: google_url(make_query(r, 'onthemarket.com')), axis=1)
review['bing_search_url'] = review.apply(lambda r: bing_url(make_query(r)), axis=1)

# Manual fields. These are intended to be edited in the CSV/Sheet by you.
manual_cols = {
    'manual_agent_name': '',
    'source_url': '',
    'evidence_type': '',
    'confidence': '',
    'review_status': 'pending',
    'review_notes': ''
}
for c, default in manual_cols.items():
    if c not in review.columns:
        review[c] = default

queue_cols = [
    'transaction_key', 'sale_year', 'date_of_transfer', 'price', 'postcode', 'postcode_district',
    'property_type', 'new_build_flag', 'tenure_type',
    'address_text',
    'google_search_url', 'rightmove_search_url', 'zoopla_search_url', 'onthemarket_search_url', 'bing_search_url',
    'manual_agent_name', 'source_url', 'evidence_type', 'confidence', 'review_status', 'review_notes'
]
queue_cols = [c for c in queue_cols if c in review.columns]

review_queue = review[queue_cols].copy()
review_queue['date_of_transfer'] = pd.to_datetime(review_queue['date_of_transfer']).dt.strftime('%Y-%m-%d')

queue_path = REPORTS / 'manual_review_queue_bn_last5y.csv'
review_queue.to_csv(queue_path, index=False)

print('Saved manual review queue:')
print(queue_path)
display(review_queue.head(10))

Saved manual review queue:
/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.csv


,transaction_key,sale_year,date_of_transfer,price,postcode,postcode_district,property_type,address_text,google_search_url,rightmove_search_url,zoopla_search_url,onthemarket_search_url,bing_search_url,manual_agent_name,source_url,evidence_type,confidence,review_status,review_notes
0,400000|2021-09-24|BN1 3TX|5|<NA>|BELMONT COURT,2021,2021-09-24,400000,BN1 3TX,BN1,F,"5, BELMONT COURT, BRIGHTON, BRIGHTON AND HOVE,...",https://www.google.com/search?q=%225%2C+BELMON...,https://www.google.com/search?q=%225%2C+BELMON...,https://www.google.com/search?q=%225%2C+BELMON...,https://www.google.com/search?q=%225%2C+BELMON...,https://www.bing.com/search?q=%225%2C+BELMONT+...,,,,,pending,
1,295000|2021-09-16|BN10 7NE|120|<NA>|CENTRAL AV...,2021,2021-09-16,295000,BN10 7NE,BN10,T,"120, CENTRAL AVENUE, TELSCOMBE CLIFFS, PEACEHA...",https://www.google.com/search?q=%22120%2C+CENT...,https://www.google.com/search?q=%22120%2C+CENT...,https://www.google.com/search?q=%22120%2C+CENT...,https://www.google.com/search?q=%22120%2C+CENT...,https://www.bing.com/search?q=%22120%2C+CENTRA...,,,,,pending,
2,"180000|2021-09-17|BN11 3HT|CAMBRIDGE LODGE, 10...",2021,2021-09-17,180000,BN11 3HT,BN11,F,"CAMBRIDGE LODGE, 10, FLAT 7, SOUTHEY ROAD, WOR...",https://www.google.com/search?q=%22CAMBRIDGE+L...,https://www.google.com/search?q=%22CAMBRIDGE+L...,https://www.google.com/search?q=%22CAMBRIDGE+L...,https://www.google.com/search?q=%22CAMBRIDGE+L...,https://www.bing.com/search?q=%22CAMBRIDGE+LOD...,,,,,pending,
3,490000|2021-06-24|BN12 6QA|55|<NA>|LANGBURY LANE,2021,2021-06-24,490000,BN12 6QA,BN12,T,"55, LANGBURY LANE, FERRING, WORTHING, ARUN, WE...",https://www.google.com/search?q=%2255%2C+LANGB...,https://www.google.com/search?q=%2255%2C+LANGB...,https://www.google.com/search?q=%2255%2C+LANGB...,https://www.google.com/search?q=%2255%2C+LANGB...,https://www.bing.com/search?q=%2255%2C+LANGBUR...,,,,,pending,
4,306000|2021-06-25|BN13 1LZ|32|<NA>|STONEHURST ...,2021,2021-06-25,306000,BN13 1LZ,BN13,T,"32, STONEHURST ROAD, WORTHING, WORTHING, WEST ...",https://www.google.com/search?q=%2232%2C+STONE...,https://www.google.com/search?q=%2232%2C+STONE...,https://www.google.com/search?q=%2232%2C+STONE...,https://www.google.com/search?q=%2232%2C+STONE...,https://www.bing.com/search?q=%2232%2C+STONEHU...,,,,,pending,
5,155000|2021-08-13|BN14 8DD|2|FLAT 4|KING EDWAR...,2021,2021-08-13,155000,BN14 8DD,BN14,F,"2, FLAT 4, KING EDWARD AVENUE, WORTHING, WORTH...",https://www.google.com/search?q=%222%2C+FLAT+4...,https://www.google.com/search?q=%222%2C+FLAT+4...,https://www.google.com/search?q=%222%2C+FLAT+4...,https://www.google.com/search?q=%222%2C+FLAT+4...,https://www.bing.com/search?q=%222%2C+FLAT+4%2...,,,,,pending,
6,500000|2021-09-17|BN15 0HD|107|<NA>|MANOR ROAD,2021,2021-09-17,500000,BN15 0HD,BN15,D,"107, MANOR ROAD, LANCING, ADUR, WEST SUSSEX, B...",https://www.google.com/search?q=%22107%2C+MANO...,https://www.google.com/search?q=%22107%2C+MANO...,https://www.google.com/search?q=%22107%2C+MANO...,https://www.google.com/search?q=%22107%2C+MANO...,https://www.bing.com/search?q=%22107%2C+MANOR+...,,,,,pending,
7,267000|2021-03-30|BN16 3PA|5|<NA>|THE STREET,2021,2021-03-30,267000,BN16 3PA,BN16,S,"5, THE STREET, RUSTINGTON, LITTLEHAMPTON, ARUN...",https://www.google.com/search?q=%225%2C+THE+ST...,https://www.google.com/search?q=%225%2C+THE+ST...,https://www.google.com/search?q=%225%2C+THE+ST...,https://www.google.com/search?q=%225%2C+THE+ST...,https://www.bing.com/search?q=%225%2C+THE+STRE...,,,,,pending,
8,585000|2021-10-27|BN17 7DA|15|<NA>|PEACHEY WAY,2021,2021-10-27,585000,BN17 7DA,BN17,D,"15, PEACHEY WAY, LITTLEHAMPTON, ARUN, WEST SUS...",https://www.google.com/search?q=%2215%2C+PEACH...,https://www.google.com/search?q=%2215%2C+PEACH...,https://www.google.com/search?q=%2215%2C+PEACH...,https://www.google.com/search?q=%2215%2C+PEACH...,https://www.bing.com/search?q=%2215%2C+PEACHEY...,,,,,pending,
9,350000|2021-05-28|BN18 0GJ|9|<NA>|LINSEED WAY,2021,2021-05-28,350000,BN18 0GJ,

## Manual review step

Open this CSV in Google Sheets or Excel:

`/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.csv`

For each row you can fill:
- `manual_agent_name`
- `source_url`
- `evidence_type`
- `confidence` as `high`, `medium`, `low`, or `unknown`
- `review_status` as `confirmed`, `candidate`, `rejected`, `unknown`, or `pending`
- `review_notes`

Then save/export it back to the same CSV path and run the remaining cells.

In [7]:
# --- 7. Re-load manual review queue after edits ---

# You can run this cell immediately before manual edits too; it will just show all pending.
edited_path = REPORTS / 'manual_review_queue_bn_last5y.csv'
manual = pd.read_csv(edited_path, dtype=str).fillna('')

# Normalise manual fields
for c in ['manual_agent_name', 'source_url', 'evidence_type', 'confidence', 'review_status', 'review_notes']:
    if c not in manual.columns:
        manual[c] = ''

manual['confidence'] = manual['confidence'].str.lower().str.strip()
manual['review_status'] = manual['review_status'].str.lower().str.strip()
manual['manual_agent_name'] = manual['manual_agent_name'].str.strip()
manual['source_url'] = manual['source_url'].str.strip()

# Valid values
valid_conf = {'', 'high', 'medium', 'low', 'unknown'}
valid_status = {'', 'pending', 'confirmed', 'candidate', 'rejected', 'unknown'}

bad_conf = manual[~manual['confidence'].isin(valid_conf)]
bad_status = manual[~manual['review_status'].isin(valid_status)]

print('Rows loaded:', len(manual))
print('Bad confidence rows:', len(bad_conf))
print('Bad status rows:', len(bad_status))

if len(bad_conf):
    display(bad_conf[['transaction_key', 'confidence']].head(20))
if len(bad_status):
    display(bad_status[['transaction_key', 'review_status']].head(20))

display(
    manual.groupby(['review_status', 'confidence'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['review_status', 'confidence'])
)

Rows loaded: 250
Bad confidence rows: 0
Bad status rows: 0


,review_status,confidence,rows
0,pending,,250


In [8]:
# --- 8. Build attribution candidates from manual evidence ---

# Candidate rows require either an agent name or explicit unknown/rejected status.
candidates = manual.copy()

# Assign useful machine flags
candidates['has_agent_name'] = candidates['manual_agent_name'].str.len() > 0
candidates['has_source_url'] = candidates['source_url'].str.len() > 0

def verdict(row):
    status = row.get('review_status', '')
    conf = row.get('confidence', '')
    has_agent = bool(row.get('has_agent_name', False))
    has_source = bool(row.get('has_source_url', False))
    if status == 'confirmed' and has_agent and has_source and conf in {'high', 'medium'}:
        return 'usable_confirmed'
    if status == 'candidate' and has_agent:
        return 'usable_candidate'
    if status in {'unknown', 'rejected'}:
        return status
    return 'pending_or_incomplete'

candidates['agent_verdict'] = candidates.apply(verdict, axis=1)
candidates['checked_utc'] = datetime.now(timezone.utc).isoformat()

candidate_cols = [
    'transaction_key', 'sale_year', 'date_of_transfer', 'price', 'postcode', 'postcode_district',
    'address_text', 'manual_agent_name', 'source_url', 'evidence_type',
    'confidence', 'review_status', 'agent_verdict', 'review_notes', 'checked_utc'
]
candidate_cols = [c for c in candidate_cols if c in candidates.columns]

agent_candidates = candidates[candidate_cols].copy()

candidates_path = REPORTS / 'agent_attribution_candidates_bn_last5y.csv'
agent_candidates.to_csv(candidates_path, index=False)

print('Saved candidates:')
print(candidates_path)

display(agent_candidates.head(20))
display(agent_candidates.groupby('agent_verdict').size().reset_index(name='rows'))

Saved candidates:
/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_attribution_candidates_bn_last5y.csv


,transaction_key,sale_year,date_of_transfer,price,postcode,postcode_district,address_text,manual_agent_name,source_url,evidence_type,confidence,review_status,agent_verdict,review_notes,checked_utc
0,400000|2021-09-24|BN1 3TX|5|<NA>|BELMONT COURT,2021,2021-09-24,400000,BN1 3TX,BN1,"5, BELMONT COURT, BRIGHTON, BRIGHTON AND HOVE,...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
1,295000|2021-09-16|BN10 7NE|120|<NA>|CENTRAL AV...,2021,2021-09-16,295000,BN10 7NE,BN10,"120, CENTRAL AVENUE, TELSCOMBE CLIFFS, PEACEHA...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
2,"180000|2021-09-17|BN11 3HT|CAMBRIDGE LODGE, 10...",2021,2021-09-17,180000,BN11 3HT,BN11,"CAMBRIDGE LODGE, 10, FLAT 7, SOUTHEY ROAD, WOR...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
3,490000|2021-06-24|BN12 6QA|55|<NA>|LANGBURY LANE,2021,2021-06-24,490000,BN12 6QA,BN12,"55, LANGBURY LANE, FERRING, WORTHING, ARUN, WE...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
4,306000|2021-06-25|BN13 1LZ|32|<NA>|STONEHURST ...,2021,2021-06-25,306000,BN13 1LZ,BN13,"32, STONEHURST ROAD, WORTHING, WORTHING, WEST ...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
5,155000|2021-08-13|BN14 8DD|2|FLAT 4|KING EDWAR...,2021,2021-08-13,155000,BN14 8DD,BN14,"2, FLAT 4, KING EDWARD AVENUE, WORTHING, WORTH...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
6,500000|2021-09-17|BN15 0HD|107|<NA>|MANOR ROAD,2021,2021-09-17,500000,BN15 0HD,BN15,"107, MANOR ROAD, LANCING, ADUR, WEST SUSSEX, B...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
7,267000|2021-03-30|BN16 3PA|5|<NA>|THE STREET,2021,2021-03-30,267000,BN16 3PA,BN16,"5, THE STREET, RUSTINGTON, LITTLEHAMPTON, ARUN...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
8,585000|2021-10-27|BN17 7DA|15|<NA>|PEACHEY WAY,2021,2021-10-27,585000,BN17 7DA,BN17,"15, PEACHEY WAY, LITTLEHAMPTON, ARUN, WEST SUS...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00
9,350000|2021-05-28|BN18 0GJ|9|<NA>|LINSEED WAY,2021,2021-05-28,350000,BN18 0GJ,BN18,"9, LINSEED WAY, YAPTON, ARUNDEL, ARUN, WEST SU...",,,,,pending,pending_or_incomplete,,2026-04-27T13:38:47.421956+00:00


,agent_verdict,rows
0,pending_or_incomplete,250


In [9]:
# --- 9. Summary reports ---

# Summary by year/status
summary_by_year = (
    agent_candidates
    .groupby(['sale_year', 'agent_verdict'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['sale_year', 'agent_verdict'])
)

# Summary by postcode district/status
summary_by_district = (
    agent_candidates
    .groupby(['postcode_district', 'agent_verdict'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['postcode_district', 'agent_verdict'])
)

# Summary by agent for usable entries
usable_agents = agent_candidates[
    agent_candidates['agent_verdict'].isin(['usable_confirmed', 'usable_candidate'])
].copy()

if len(usable_agents):
    summary_by_agent = (
        usable_agents
        .groupby(['manual_agent_name', 'confidence'], dropna=False)
        .agg(
            rows=('transaction_key', 'count'),
            min_year=('sale_year', 'min'),
            max_year=('sale_year', 'max')
        )
        .reset_index()
        .sort_values(['rows', 'manual_agent_name'], ascending=[False, True])
    )
else:
    summary_by_agent = pd.DataFrame(columns=['manual_agent_name', 'confidence', 'rows', 'min_year', 'max_year'])

summary_by_year.to_csv(REPORTS / 'agent_summary_by_year_bn_last5y.csv', index=False)
summary_by_district.to_csv(REPORTS / 'agent_summary_by_postcode_district_bn_last5y.csv', index=False)
summary_by_agent.to_csv(REPORTS / 'agent_summary_by_agent_bn_last5y.csv', index=False)

print('Saved summaries to:', REPORTS)
print('\nBy year:')
display(summary_by_year)
print('\nBy postcode district:')
display(summary_by_district.head(50))
print('\nBy agent:')
display(summary_by_agent.head(50))

Saved summaries to: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution

By year:


,sale_year,agent_verdict,rows
0,2021,pending_or_incomplete,50
1,2022,pending_or_incomplete,50
2,2023,pending_or_incomplete,50
3,2024,pending_or_incomplete,50
4,2025,pending_or_incomplete,50



By postcode district:


,postcode_district,agent_verdict,rows
0,BN1,pending_or_incomplete,10
1,BN10,pending_or_incomplete,9
2,BN11,pending_or_incomplete,7
3,BN12,pending_or_incomplete,10
4,BN13,pending_or_incomplete,9
5,BN14,pending_or_incomplete,10
6,BN15,pending_or_incomplete,12
7,BN16,pending_or_incomplete,8
8,BN17,pending_or_incomplete,8
9,BN18,pending_or_incomplete,8



By agent:


,manual_agent_name,confidence,rows,min_year,max_year


In [10]:
# --- 10. Write manifest for provenance ---

manifest = {
    'project': 'Addresses26',
    'notebook': 'Addresses26_03_agent_attribution_BN_last5y_v4_manual_review.ipynb',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'postcode_prefix': POSTCODE_PREFIX,
    'years': YEARS,
    'sample_per_year': SAMPLE_PER_YEAR,
    'balanced_by_district': BALANCE_BY_DISTRICT,
    'input_parquet_files': [str(p) for p in parquet_files],
    'output_files': {
        'manual_review_queue': str(queue_path),
        'agent_candidates': str(candidates_path),
        'summary_by_year': str(REPORTS / 'agent_summary_by_year_bn_last5y.csv'),
        'summary_by_postcode_district': str(REPORTS / 'agent_summary_by_postcode_district_bn_last5y.csv'),
        'summary_by_agent': str(REPORTS / 'agent_summary_by_agent_bn_last5y.csv'),
    },
    'method_note': (
        'Supervised manual evidence workflow. Land Registry Price Paid Data remains the factual core. '
        'Estate-agent attribution is a separate probabilistic enrichment layer based on manually reviewed public evidence.'
    ),
}

manifest_path = REPORTS / 'agent_attribution_bn_last5y_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved manifest:')
print(manifest_path)
print(json.dumps(manifest, indent=2)[:2000])

Saved manifest:
/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_attribution_bn_last5y_manifest.json
{
  "project": "Addresses26",
  "notebook": "Addresses26_03_agent_attribution_BN_last5y_v4_manual_review.ipynb",
  "created_utc": "2026-04-27T13:38:47.652081+00:00",
  "postcode_prefix": "BN",
  "years": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "sample_per_year": 50,
  "balanced_by_district": true,
  "input_parquet_files": [
    "/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/pp_bn_standard_2021.parquet",
    "/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/pp_bn_standard_2022.parquet",
    "/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/pp_bn_standard_2023.parquet",
    "/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/pp_bn_standard_2024.parquet",
    "/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid/pp_bn_standard_2025.pa

In [11]:
# --- 11. Final verdict helper ---

total = len(agent_candidates)
counts = agent_candidates['agent_verdict'].value_counts().to_dict()
usable = counts.get('usable_confirmed', 0) + counts.get('usable_candidate', 0)

print('--- AGENT ATTRIBUTION PILOT VERDICT ---')
print('Total review rows:', total)
print('Usable confirmed/candidate rows:', usable)
print('Pending/incomplete rows:', counts.get('pending_or_incomplete', 0))
print('Unknown rows:', counts.get('unknown', 0))
print('Rejected rows:', counts.get('rejected', 0))

if total:
    print('Usable rate: {:.1f}%'.format(100 * usable / total))

print('\nRecommendation:')
if usable == 0:
    print('Manual review queue created. Fill agent evidence fields, then re-run cells 7 onward.')
elif usable / max(total, 1) < 0.25:
    print('The workflow is viable but evidence coverage is still low. Continue manual review or narrow to high-value districts.')
else:
    print('The workflow is producing useful coverage. Consider expanding sample size or moving to district-specific batches.')

--- AGENT ATTRIBUTION PILOT VERDICT ---
Total review rows: 250
Usable confirmed/candidate rows: 0
Pending/incomplete rows: 250
Unknown rows: 0
Rejected rows: 0
Usable rate: 0.0%

Recommendation:
Manual review queue created. Fill agent evidence fields, then re-run cells 7 onward.
